# 07 — Model Comparison & Selection

Side-by-side comparison of every backtested model on the leakage-safe walk-forward output
(`reports/predictions.parquet`): probabilistic metrics (log loss / Brier), classification
results (confusion matrix + precision / recall / F1 / accuracy), and reliability. Generalises
`04_model_accuracy.ipynb`, which covers `rf_tuned` alone.

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.calibration import calibration_curve

# Make the project root importable (this notebook lives in notebooks/)
ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent

# Shared conventions (match notebooks 03/04/05/06)
LABEL_ORDER = ["HW", "D", "AW"]
LABEL_NAMES = {"HW": "Home win", "D": "Draw", "AW": "Away win"}
PALETTE = {"HW": "#2a9d8f", "D": "#e9c46a", "AW": "#e76f51"}
PROB_COLS = ["p_home", "p_draw", "p_away"]
CLASS_IDX = {l: i for i, l in enumerate(LABEL_ORDER)}

preds = pd.read_parquet(ROOT / "reports" / "predictions.parquet")

# Compare every model, baseline-first; append any not in the preferred order.
ORDER = ["baseline_elo", "log_reg", "rf", "rf_tuned", "rf_recency", "rf_calibrated"]
available = list(preds["model"].unique())
MODELS = [m for m in ORDER if m in available] + [m for m in available if m not in ORDER]

def model_frame(m):
    return preds[preds["model"] == m].reset_index(drop=True)

def predictions(df):
    P = df[PROB_COLS].to_numpy()
    return df["y_true"].to_numpy(), P, np.array(LABEL_ORDER)[P.argmax(axis=1)]

print("models compared:", MODELS)
print("backtested matches per model:", len(model_frame(MODELS[0])))

## 1. Probabilistic metrics — log loss & Brier

In [ ]:
# Multiclass log loss + Brier for a model's prediction rows.
def multiclass_metrics(df):
    Pm = df[PROB_COLS].to_numpy()
    yi = df["y_true"].map(CLASS_IDX).to_numpy()
    ll = -np.mean(np.log(np.clip(Pm[np.arange(len(Pm)), yi], 1e-15, None)))
    onehot = np.zeros_like(Pm); onehot[np.arange(len(Pm)), yi] = 1.0
    brier = np.mean(np.sum((Pm - onehot) ** 2, axis=1))
    return ll, brier

summary = pd.DataFrame(
    [{"model": m, **dict(zip(["log_loss", "brier"], multiclass_metrics(model_frame(m))))}
     for m in MODELS]
).sort_values("log_loss").reset_index(drop=True)

# Reference floor: always predicting the class base rates.
d0 = model_frame(MODELS[0])
yi_all = d0["y_true"].map(CLASS_IDX).to_numpy()
base = np.bincount(yi_all, minlength=3) / len(yi_all)
base_ll = -np.mean(np.log(base[yi_all]))

print(summary.round(4).to_string(index=False))
print(f"\nbase-rate log loss: {base_ll:.4f}   uniform (1/3): {np.log(3):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for ax, metric, title in zip(axes, ["log_loss", "brier"], ["Log loss", "Brier"]):
    s = summary.sort_values(metric)
    ax.bar(s["model"], s[metric], color="#9aa6ad")
    ax.set_title(f"{title} by model (lower is better)")
    ax.tick_params(axis="x", rotation=45)
axes[0].axhline(base_ll, color="#e76f51", ls="--", label=f"base-rate ({base_ll:.3f})")
axes[0].legend()
plt.tight_layout(); plt.show()

## 2. Classification — confusion matrix & precision / recall / F1

In [ ]:
# Accuracy overview across models (+ draw recall and how many draws each model ever picks).
acc_rows = []
for m in MODELS:
    yt, P, yp = predictions(model_frame(m))
    acc_rows.append({
        "model": m,
        "accuracy": (yp == yt).mean(),
        "draw_recall": ((yp == "D") & (yt == "D")).sum() / max((yt == "D").sum(), 1),
        "draws_predicted": int((yp == "D").sum()),
    })
print(pd.DataFrame(acc_rows).round(4).to_string(index=False))
print(f"\nalways-home baseline accuracy: {(d0['y_true'] == 'HW').mean():.4f}")

In [ ]:
# Confusion matrix per model.
ncols = 3
nrows = int(np.ceil(len(MODELS) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(13, 4 * nrows))
for ax, m in zip(axes.ravel(), MODELS):
    yt, P, yp = predictions(model_frame(m))
    cm = confusion_matrix(yt, yp, labels=LABEL_ORDER)
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(3)); ax.set_xticklabels(LABEL_ORDER)
    ax.set_yticks(range(3)); ax.set_yticklabels(LABEL_ORDER)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_title(m)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() * 0.6 else "black")
for ax in axes.ravel()[len(MODELS):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Per-class precision / recall / F1 / accuracy for every model.
for m in MODELS:
    yt, P, yp = predictions(model_frame(m))
    print(f"===== {m} =====")
    print(classification_report(
        yt, yp, labels=LABEL_ORDER,
        target_names=[LABEL_NAMES[l] for l in LABEL_ORDER], zero_division=0,
    ))

## 3. Reliability (calibration)

In [ ]:
# Per-outcome (one-vs-rest) reliability for each model; points on the diagonal = calibrated.
for m in MODELS:
    dm = model_frame(m)
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
    for ax, label, col in zip(axes, LABEL_ORDER, PROB_COLS):
        y_bin = (dm["y_true"] == label).astype(int).to_numpy()
        frac_pos, mean_pred = calibration_curve(
            y_bin, dm[col].to_numpy(), n_bins=10, strategy="quantile"
        )
        ax.plot(mean_pred, frac_pos, "o-", color=PALETTE[label])
        ax.plot([0, 1], [0, 1], "k:", alpha=0.6)
        ax.set_title(LABEL_NAMES[label]); ax.set_xlabel("Mean predicted prob")
    axes[0].set_ylabel("Observed frequency")
    fig.suptitle(f"Reliability — {m}")
    plt.tight_layout(); plt.show()